In [1]:
import os
import html

import pandas as pd
import geopandas as gpd

import matplotlib.pyplot as plt
import contextily as ctx

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

In [2]:
# -----------------------
# Files
# -----------------------
data_file = "validation_is_informative_orphan_source.gpkg"
label_file = "validation_is_informative_orphan_output.csv"
layer_name = "validation_is_informative_orphan"   # layer name inside the gpkg

# -----------------------
# Load source data (GeoPackage is EPSG:27700)
# -----------------------
gdf = gpd.read_file(data_file, layer=layer_name).copy()

# Ensure CRS is set correctly
if gdf.crs is None:
    gdf = gdf.set_crs("EPSG:27700")

# Ensure decision_date is datetime
if "decision_date" in gdf.columns:
    gdf["decision_date"] = pd.to_datetime(gdf["decision_date"], errors="coerce", dayfirst=True)

# Ensure IDs are strings (if present)
if "lpa_app_no" in gdf.columns:
    gdf["lpa_app_no"] = gdf["lpa_app_no"].astype(str)

# Required columns
required_cols = ["geometry", "id", "child_group"]
missing = [c for c in required_cols if c not in gdf.columns]
if missing:
    raise ValueError(f"Missing required columns in source: {missing}")

In [3]:
# Drop problematic group (child_group == 41.0)
if "child_group" in gdf.columns:
    gdf = gdf[~(gdf["child_group"] == 41.0)].copy()

In [4]:
# Drop problematic uid
if "child_group" in gdf.columns:
    gdf = gdf[~(gdf["id"] == 411)].copy()

In [5]:
# gdf.columns.tolist()

In [6]:
# -----------------------
# Create output file if missing
# Output must include: uid + useful_orphan + helper columns
# -----------------------
out_cols = [
    "id",
    "child_group",
    "useful_orphan",
    "lpa_app_no",
    "lpa_name",
    "decision_date",
    "description",
    "decision",
    "url_planning_app",
]

if not os.path.exists(label_file):
    pd.DataFrame(columns=out_cols).to_csv(label_file, index=False)

# Load existing validations (DO NOT override; append only)
validated = pd.read_csv(label_file, dtype={"uid": str})
validated_uids = set(validated["uid"].dropna().astype(str))

# Only keep rows not already validated (by uid)
to_label = gdf[~gdf["uid"].astype(str).isin(validated_uids)].copy()

# -----------------------
# Grouping logic
# - group by child_group when not NaN
# - if child_group is NaN, treat each row as its own group
# -----------------------
def _group_key(row):
    cg = row.get("child_group", None)
    if pd.isna(cg):
        # Unique group per row
        return ("__nan__", str(row["uid"]))
    return ("child_group", str(cg))

to_label["_group_key"] = to_label.apply(_group_key, axis=1)

# Make an ordered group list (stable)
group_keys = to_label["_group_key"].dropna().tolist()
# preserve first-seen order
seen = set()
group_order = []
for k in group_keys:
    if k not in seen:
        seen.add(k)
        group_order.append(k)

# -----------------------
# Helpers
# -----------------------
def append_validations(rows_to_write):
    """Append multiple validated rows to CSV immediately (no overwrite)."""
    if not rows_to_write:
        return
    out_df = pd.DataFrame(rows_to_write, columns=out_cols)
    out_df.to_csv(label_file, mode="a", header=False, index=False)

def _to_web_mercator(gdf_to_plot):
    if gdf_to_plot.crs is None:
        gdf_to_plot = gdf_to_plot.set_crs("EPSG:27700", allow_override=True)
    return gdf_to_plot.to_crs(epsg=3857)

def _safe_str(v):
    if v is None:
        return ""
    try:
        if pd.isna(v):
            return ""
    except Exception:
        pass
    if isinstance(v, pd.Timestamp):
        if pd.isna(v):
            return ""
        return v.strftime("%Y-%m-%d")
    return str(v)

def _render_map(one_row_gdf):
    to_plot_web = _to_web_mercator(one_row_gdf)

    fig, ax = plt.subplots(figsize=(4.6, 4.6))
    to_plot_web.boundary.plot(ax=ax)

    minx, miny, maxx, maxy = to_plot_web.total_bounds
    if not all(pd.notna([minx, miny, maxx, maxy])) or (minx == maxx) or (miny == maxy):
        ax.autoscale()
    else:
        pad_x = (maxx - minx) * 0.2
        pad_y = (maxy - miny) * 0.2
        ax.set_xlim(minx - pad_x, maxx + pad_x)
        ax.set_ylim(miny - pad_y, maxy + pad_y)

    ctx.add_basemap(
        ax,
        crs=to_plot_web.crs,
        source=ctx.providers.CartoDB.Positron,
        zoom=18
    )
    ax.set_axis_off()
    plt.show()

def _row_card_html(row_dict):
    # Include the fields you requested:
    uid = html.escape(_safe_str(row_dict.get("id", "")))
    child_group = html.escape(_safe_str(row_dict.get("child_group", "")))
    lpa_name = html.escape(_safe_str(row_dict.get("lpa_name", "")))
    decision_date = html.escape(_safe_str(row_dict.get("decision_date", "")))
    decision = html.escape(_safe_str(row_dict.get("decision", "")))
    url = _safe_str(row_dict.get("url_planning_app", ""))
    url_esc = html.escape(url)

    desc_raw = _safe_str(row_dict.get("description", ""))
    desc = html.escape(desc_raw).replace("\n", "<br>")

    # Link if present
    url_html = f"<a href='{url_esc}' target='_blank' rel='noopener noreferrer'>{url_esc}</a>" if url else ""

    return f"""
    <div style="border:1px solid #ddd; padding:8px; border-radius:6px;">
      <div><b>uid:</b> {uid}</div>
      <div><b>child_group:</b> {child_group}</div>
      <div style="margin-top:6px;"><b>lpa_name:</b> {lpa_name}</div>
      <div><b>decision_date:</b> {decision_date}</div>
      <div><b>decision:</b> {decision}</div>
      <div><b>url_planning_app:</b> {url_html}</div>
      <div style="margin-top:8px;"><b>Description:</b></div>
      <div style="max-height:240px; overflow:auto; border:1px solid #ccc; padding:8px; border-radius:6px; white-space:normal;">
        {desc}
      </div>
    </div>
    """

def _selection_status_html(group_rows, decisions):
    lines = []
    for _, r in group_rows.iterrows():
        uid = str(r["id"])
        status = decisions.get(uid, None)
        if status is True:
            s = "✅ useful"
        elif status is False:
            s = "❌ not useful"
        else:
            s = "⬜ not set"
        lines.append(f"<li><b>{html.escape(uid)}</b>: {s}</li>")
    return "<ul style='margin:6px 0 0 18px;'>" + "".join(lines) + "</ul>"

def _all_set(group_rows, decisions):
    return all(str(r["id"]) in decisions for _, r in group_rows.iterrows())

def _group_title(key):
    kind, val = key
    if kind == "__nan__":
        return "child_group = NaN (single row)"
    return f"child_group = {val}"

# -----------------------
# Widgets + state
# -----------------------
output = widgets.Output()
header = widgets.HTML()

next_group_btn = widgets.Button(description="Next ▶", button_style="info", disabled=True)
skip_group_btn = widgets.Button(description="Skip ⏭️", button_style="warning")
select_all_btn = widgets.Button(description="Select all ✅", button_style="success", disabled=True)
select_none_btn = widgets.Button(description="Select none ❌", button_style="danger", disabled=True)

state = {
    "group_idx": 0,
    "decisions": {}  # {uid: True/False}
}

def show_group():
    with output:
        clear_output()

        if state["group_idx"] >= len(group_order):
            header.value = "<h3>🎉 All orphan groups have been validated (or skipped).</h3>"
            next_group_btn.disabled = True
            skip_group_btn.disabled = True
            select_all_btn.disabled = True
            select_none_btn.disabled = True
            return

        gk = group_order[state["group_idx"]]
        group_rows = to_label[to_label["_group_key"] == gk].copy()

        decisions = state["decisions"]

        header.value = f"""
        <h3>Group {state['group_idx']+1} / {len(group_order)} — {_group_title(gk)}</h3>
        <div>Rows in group: <b>{len(group_rows)}</b></div>
        <div style="margin-top:6px;"><b>Selections:</b>{_selection_status_html(group_rows, decisions)}</div>
        """

        # Enable batch buttons only if more than 1 row
        select_all_btn.disabled = (len(group_rows) <= 1)
        select_none_btn.disabled = (len(group_rows) <= 1)

        # enable/disable Next button depending on completion
        next_group_btn.disabled = not _all_set(group_rows, decisions)

        # Render each row
        for _, r in group_rows.iterrows():
            uid = str(r["id"])
            app_no = _safe_str(r.get("lpa_app_no", ""))

            useful_btn = widgets.Button(description="Useful ✅", button_style="success")
            not_useful_btn = widgets.Button(description="Not useful ❌", button_style="danger")

            def _make_onclick(uid, val):
                def _cb(_):
                    state["decisions"][uid] = val
                    show_group()
                return _cb

            useful_btn.on_click(_make_onclick(uid, True))
            not_useful_btn.on_click(_make_onclick(uid, False))

            map_out = widgets.Output(layout=widgets.Layout(width="34%"))
            info_html = widgets.HTML(value=_row_card_html(r.to_dict()), layout=widgets.Layout(width="46%"))
            btn_box = widgets.VBox([useful_btn, not_useful_btn], layout=widgets.Layout(width="20%"))

            row_box = widgets.HBox([map_out, info_html, btn_box])

            display(HTML(
                f"<hr>"
                f"<div><b>uid:</b> {html.escape(uid)}"
                f"{' &nbsp; | &nbsp; <b>lpa_app_no:</b> ' + html.escape(app_no) if app_no else ''}"
                f"</div>"
            ))
            display(row_box)

            with map_out:
                clear_output()
                one = gpd.GeoDataFrame([r], geometry="geometry", crs=gdf.crs)
                _render_map(one)

def on_select_all(_):
    if state["group_idx"] >= len(group_order):
        return
    gk = group_order[state["group_idx"]]
    group_rows = to_label[to_label["_group_key"] == gk]
    for _, r in group_rows.iterrows():
        state["decisions"][str(r["id"])] = True
    show_group()

def on_select_none(_):
    if state["group_idx"] >= len(group_order):
        return
    gk = group_order[state["group_idx"]]
    group_rows = to_label[to_label["_group_key"] == gk]
    for _, r in group_rows.iterrows():
        state["decisions"][str(r["id"])] = False
    show_group()

def on_next_group(_):
    if state["group_idx"] >= len(group_order):
        return

    gk = group_order[state["group_idx"]]
    group_rows = to_label[to_label["_group_key"] == gk].copy()

    if not _all_set(group_rows, state["decisions"]):
        return

    rows_to_write = []
    for _, r in group_rows.iterrows():
        uid = str(r["id"])
        rows_to_write.append({
            "id": id,
            "child_group": _safe_str(r.get("child_group", "")),
            "useful_orphan": bool(state["decisions"][uid]),
            "lpa_app_no": _safe_str(r.get("lpa_app_no", "")),
            "lpa_name": _safe_str(r.get("lpa_name", "")),
            "decision_date": _safe_str(r.get("decision_date", "")),
            "description": _safe_str(r.get("description", "")),
            "decision": _safe_str(r.get("decision", "")),
            "url_planning_app": _safe_str(r.get("url_planning_app", "")),
        })

    append_validations(rows_to_write)

    # advance
    state["group_idx"] += 1
    state["decisions"] = {}
    show_group()

def on_skip_group(_):
    state["group_idx"] += 1
    state["decisions"] = {}
    show_group()

next_group_btn.on_click(on_next_group)
skip_group_btn.on_click(on_skip_group)
select_all_btn.on_click(on_select_all)
select_none_btn.on_click(on_select_none)

# -----------------------
# Display UI
# -----------------------
display(header)
display(widgets.HBox([next_group_btn, skip_group_btn, select_all_btn, select_none_btn]))
display(output)

show_group()

HTML(value='')

Output()